In [1]:
%reload_ext autoreload
%autoreload 2

# Imports

In [ ]:
from kret_notebook import *  # NOTE import first
from kret_lgbm._core.lgbm_nb_imports import *
from kret_lightning._core.lightning_nb_imports import *
from kret_matplotlib._core.mpl_nb_imports import *
from kret_np_pd._core.np_pd_nb_imports import *

# from kret_optuna._core.optuna_nb_imports import *
# from kret_polars._core.polars_nb_imports import *
# from kret_rosetta._core.rosetta_nb_imports import *
# from kret_sklearn._core.sklearn_nb_imports import *
# from kret_torch_utils._core.torch_nb_imports import *
# from kret_tqdm._core.tqdm_nb_imports import *
# from kret_type_hints._core.types_nb_imports import *
from kret_utils._core.utils_nb_imports import *

# from kret_wandb._core.wandb_nb_imports import *  # NOTE this is slow to import

Loaded environment variables from /Users/Akseldkw/coding/kretsinger/.env
[kret_lgbm._core.lgbm_nb_imports] Imported kret_lgbm._core.lgbm_nb_imports in 1.9396 seconds
[kret_lightning._core.lightning_nb_imports] Imported kret_lightning._core.lightning_nb_imports in 3.9563 seconds
[kret_matplotlib._core.mpl_nb_imports] Imported kret_matplotlib._core.mpl_nb_imports in 0.3964 seconds
[kret_np_pd._core.np_pd_nb_imports] Imported kret_np_pd._core.np_pd_nb_imports in 0.0005 seconds
[kret_optuna._core.optuna_nb_imports] Imported kret_optuna._core.optuna_nb_imports in 0.0007 seconds
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0000 seconds
[kret_rosetta._core.rosetta_nb_imports] Imported kret_rosetta._core.rosetta_nb_imports in 0.0000 seconds
[kret_sklearn._core.sklearn_nb_imports] Imported kret_sklearn._core.sklearn_nb_imports in 0.0875 seconds
[kret_torch_utils._core.torch_nb_imports] Imported kret_torch_utils._core.torch_nb_imports in 0.4276 seconds


# Load Data

In [ ]:
df = pd.DataFrame(
    {
        "gameId": [1, 1, 2, 2, 3],
        "period": [1, 2, 1, 2, 1],
        "time_remaining": [600, 200, 580, 100, 700],
        "time_elapsed": [120, 520, 140, 620, 20],
        "score_home": [10, 22, 8, 25, 15],
        "score_away": [11, 20, 12, 23, 14],
        "player_id": [101, 102, 103, 104, 105],
        "player_name": ["A", "B", "C", "D", "E"],
    }
)
styler = df.style.background_gradient(subset=["score_home", "score_away"], cmap="Blues")
df.shape, list(df.columns)

((5, 8),
 ['gameId',
  'period',
  'time_remaining',
  'time_elapsed',
  'score_home',
  'score_away',
  'player_id',
  'player_name'])

# Implementation

## Direct `col_filter` on a DataFrame

In [4]:
# include: keep only columns whose name contains any of the substrings
UKS_NP_PD.col_filter(df, include=["score", "time"])

Returning df without 4 columns: ['gameId', 'period', 'player_id', 'player_name']


,time_remaining,time_elapsed,score_home,score_away
0,600,120,10,11
1,200,520,22,20
2,580,140,8,12
3,100,620,25,23
4,700,20,15,14


In [5]:
# exclude: drop columns whose name contains any of the substrings
UKS_NP_PD.col_filter(df, exclude=["player", "period"])

Returning df without 3 columns: ['period', 'player_id', 'player_name']


,gameId,time_remaining,time_elapsed,score_home,score_away
0,1,600,120,10,11
1,1,200,520,22,20
2,2,580,140,8,12
3,2,100,620,25,23
4,3,700,20,15,14


In [6]:
# both: include first, then exclude. `time_elapsed` matches `time` (kept), but `elapsed` drops it.
UKS_NP_PD.col_filter(df, include=["score", "time"], exclude=["elapsed"])

Returning df without 5 columns: ['gameId', 'period', 'time_elapsed', 'player_id', 'player_name']


,time_remaining,score_home,score_away
0,600,10,11
1,200,22,20
2,580,8,12
3,100,25,23
4,700,15,14


## On a `Styler`: hidden via `Styler.hide(axis="columns")` so styling survives

In [7]:
# Styler input → Styler output. Columns are HIDDEN, not dropped: the underlying
# data still has every column, so any styling rules tied to a column survive.
filtered = UKS_NP_PD.col_filter(styler, include=["score"])
print("returned type:        ", type(filtered).__name__)
print("underlying data cols: ", list(filtered.data.columns))
filtered

Returning df without 6 columns: ['gameId', 'period', 'time_remaining', 'time_elapsed', 'player_id', 'player_name']
returned type:         Styler
underlying data cols:  ['gameId', 'period', 'time_remaining', 'time_elapsed', 'score_home', 'score_away', 'player_id', 'player_name']


,score_home,score_away
0,10,11
1,22,20
2,8,12
3,25,23
4,15,14


In [ ]:
# Sanity: gradient styling on score_home / score_away survives the column-hide.
# Each gradient cell renders as <td style="background-color: ...">, so the count is preserved.
import re


def gradient_cells(s):
    return len(re.findall(r"background-color:", s.to_html()))


print(f"original styler:                            {gradient_cells(styler)} gradient cells")
print(f"col_filter(styler, include=['score']):      {gradient_cells(filtered)} gradient cells")
print(
    f"col_filter(styler, exclude=['score_away']): {gradient_cells(UKS_NP_PD.col_filter(styler, exclude=['score_away']))} gradient cells (only score_home shown)"
)

original styler:                            8 gradient cells
col_filter(styler, include=['score']):      8 gradient cells
Returning df without 1 columns: ['score_away']
col_filter(styler, exclude=['score_away']): 5 gradient cells (only score_home shown)


## Via `dtt(..., include=[...], exclude=[...])`

In [9]:
# DataFrame input: column filter applied before HTML render.
UKS_NP_PD.dtt(df, n=3, include=["score", "time"])

Returning df without 4 columns: ['gameId', 'period', 'player_id', 'player_name']


,time_remaining,time_elapsed,score_home,score_away
,int64,int64,int64,int64
0,600,120,10,11
3,100,620,25,23
4,700,20,15,14


In [10]:
# Styler input: gradient on score_home / score_away survives, non-matching cols are hidden.
UKS_NP_PD.dtt(styler, n=3, include=["score"])

Returning df without 6 columns: ['gameId', 'period', 'time_remaining', 'time_elapsed', 'player_id', 'player_name']


,score_home,score_away
0,10,11
3,25,23
4,15,14


In [11]:
# Multiple inputs: each is independently column-filtered before display.
df2 = df.assign(extra_metric=df["score_home"] - df["score_away"])
UKS_NP_PD.dtt([df, df2], n=3, include=["score"], titles=["original", "with extra_metric"])

Returning df without 6 columns: ['gameId', 'period', 'time_remaining', 'time_elapsed', 'player_id', 'player_name']
Returning df without 7 columns: ['gameId', 'period', 'time_remaining', 'time_elapsed', 'player_id', 'player_name', 'extra_metric']


,score_home,score_away
,int64,int64
1,22,20
3,25,23
4,15,14
,score_home,score_away
,int64,int64
1,22,20
3,25,23
4,15,14


# Sandbox